# NSMC + `klue/bert-base` Fine-tuning 실습

## 목표

이 노트북은 아래 과제 조건을 충족하도록 구성되어 있습니다.

1. NSMC 데이터 분석 및 Hugging Face Dataset 구성
2. `klue/bert-base` 모델과 tokenizer 로드
3. tokenizer 기반 전처리 및 모델 학습
4. fine-tuning으로 validation accuracy 90% 이상 목표
5. bucketing + dynamic padding 적용 후 STEP 4 결과와 비교
6. 모델이 정상 작동하는지 예측 샘플로 확인

## 핵심 비교 구조

- **STEP 4:** `padding="max_length"` 고정 padding
- **STEP 5:** `DataCollatorWithPadding` + `group_by_length=True` 기반 dynamic padding/bucketing

In [1]:
# STEP 0. 패키지 설치
!pip -q install -U \
    "transformers>=4.44.0" \
    "datasets>=2.19.0" \
    "evaluate>=0.4.2" \
    "accelerate>=0.33.0" \
    "scikit-learn>=1.4.0" \
    "pandas>=2.0.0" \
    "matplotlib>=3.8.0"

In [2]:
# STEP 0-1. 라이브러리 로드 및 전역 설정

import os
import gc
import time
import json
import random
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU 환경을 권장합니다.")

torch: 2.7.1+cu118
cuda available: True
GPU: Tesla T4


In [3]:
# STEP 0-2. 하드코딩된 실험 설정값

MODEL_NAME = "klue/bert-base"
DATASET_NAME = "e9t/nsmc"

TEXT_COL = "document"
LABEL_COL = "label"

MAX_LENGTH = 128
SEED = 42

# 90% 이상을 노리는 안정적인 설정입니다.
# GPU 메모리가 부족하면 PER_DEVICE_TRAIN_BATCH_SIZE를 16으로 낮추세요.
NUM_EPOCHS = 4
LEARNING_RATE = 2e-5
PER_DEVICE_TRAIN_BATCH_SIZE = 32
PER_DEVICE_EVAL_BATCH_SIZE = 64
WARMUP_RATIO = 0.10
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 2

STATIC_OUTPUT_DIR = "./outputs/nsmc-klue-bert-static-padding"
BUCKET_OUTPUT_DIR = "./outputs/nsmc-klue-bert-bucketing"

ID2LABEL = {0: "NEGATIVE", 1: "POSITIVE"}
LABEL2ID = {"NEGATIVE": 0, "POSITIVE": 1}


def set_seed_all(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 완전 재현성을 높이기 위한 설정입니다.
    # 속도는 약간 느려질 수 있습니다.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed_all(SEED)

# STEP 1. NSMC 데이터 분석 및 Hugging Face Dataset 구성

In [4]:
# STEP 1-1. Hugging Face datasets에서 NSMC 로드

from datasets import Features, Value

NSMC_TRAIN_URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
NSMC_TEST_URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"

nsmc_features = Features({
    "id": Value("int64"),
    "document": Value("string"),
    "label": Value("int64"),
})

raw_datasets = load_dataset(
    "csv",
    data_files={
        "train": NSMC_TRAIN_URL,
        "test": NSMC_TEST_URL,
    },
    delimiter="\t",
    features=nsmc_features,
)

print(raw_datasets)
print(raw_datasets["train"][0])
print(raw_datasets["test"][0])

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 150000
    })
    test: Dataset({
        features: ['id', 'document', 'label'],
        num_rows: 50000
    })
})
{'id': 9976970, 'document': '아 더빙.. 진짜 짜증나네요 목소리', 'label': 0}
{'id': 6270596, 'document': '굳 ㅋ', 'label': 1}


In [5]:
# STEP 1-2. 데이터 기본 분석

train_df_raw = raw_datasets["train"].to_pandas()
test_df_raw = raw_datasets["test"].to_pandas()

def summarize_raw_df(df: pd.DataFrame, split_name: str) -> dict:
    doc = df[TEXT_COL]
    return {
        "split": split_name,
        "rows": len(df),
        "null_document": int(doc.isna().sum()),
        "empty_document": int(doc.fillna("").astype(str).str.strip().eq("").sum()),
        "duplicated_document": int(doc.duplicated().sum()),
        "negative_0": int((df[LABEL_COL] == 0).sum()),
        "positive_1": int((df[LABEL_COL] == 1).sum()),
        "mean_char_len": float(doc.fillna("").astype(str).str.len().mean()),
        "median_char_len": float(doc.fillna("").astype(str).str.len().median()),
        "max_char_len": int(doc.fillna("").astype(str).str.len().max()),
    }

raw_summary = pd.DataFrame([
    summarize_raw_df(train_df_raw, "train"),
    summarize_raw_df(test_df_raw, "test"),
])

display(raw_summary)

print("Train label ratio")
display(train_df_raw[LABEL_COL].value_counts(normalize=True).sort_index())

print("Test label ratio")
display(test_df_raw[LABEL_COL].value_counts(normalize=True).sort_index())

print("Sample rows")
display(train_df_raw.sample(5, random_state=SEED))

,split,rows,null_document,empty_document,duplicated_document,negative_0,positive_1,mean_char_len,median_char_len,max_char_len
0,train,150000,5,5,3817,75173,74827,35.203353,27.0,146
1,test,50000,3,3,842,24827,25173,35.318140,27.0,144


Train label ratio


label
0    0.501153
1    0.498847
Name: proportion, dtype: float64

Test label ratio


label
0    0.49654
1    0.50346
Name: proportion, dtype: float64

Sample rows


,id,document,label
59770,8932939,수OO만에 다시보네여,1
21362,3681731,일방적인 영화다. 관객 좀 고려해주시길,0
127324,9847174,세상을 초월하는 한 사람의 선한 마음,1
140509,8506899,멍하다.. 여러생각이 겹치는데 오랜만에 영화 보고 이런 느낌 느껴본다,1
144297,9991656,"우와 별 반개도 아까운판에 밑에 CJ 알바생들 쩐다.. 전부 만점이야 ㅎㅎㅎ..,....",0


In [6]:
# STEP 1-3. 결측치/공백/중복 제거 후 DatasetDict 구성

def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    cleaned = df[[TEXT_COL, LABEL_COL]].copy()
    cleaned = cleaned.dropna(subset=[TEXT_COL])
    cleaned[TEXT_COL] = cleaned[TEXT_COL].astype(str).str.strip()
    cleaned = cleaned[cleaned[TEXT_COL] != ""]
    cleaned = cleaned.drop_duplicates(subset=[TEXT_COL])
    cleaned[LABEL_COL] = cleaned[LABEL_COL].astype(int)
    cleaned = cleaned.reset_index(drop=True)
    return cleaned

train_df = clean_df(train_df_raw)
valid_df = clean_df(test_df_raw)

clean_summary = pd.DataFrame([
    summarize_raw_df(train_df, "train_cleaned"),
    summarize_raw_df(valid_df, "validation_cleaned"),
])

display(clean_summary)

datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(valid_df, preserve_index=False),
})

# Trainer가 안정적으로 labels를 인식하도록 label -> labels로 변경합니다.
datasets = datasets.rename_column(LABEL_COL, "labels")

print(datasets)
print(datasets["train"][0])

,split,rows,null_document,empty_document,duplicated_document,negative_0,positive_1,mean_char_len,median_char_len,max_char_len
0,train_cleaned,146182,0,0,0,73342,72840,35.981338,27.0,146
1,validation_cleaned,49157,0,0,0,24446,24711,35.848384,27.0,144


DatasetDict({
    train: Dataset({
        features: ['document', 'labels'],
        num_rows: 146182
    })
    validation: Dataset({
        features: ['document', 'labels'],
        num_rows: 49157
    })
})
{'document': '아 더빙.. 진짜 짜증나네요 목소리', 'labels': 0}


# STEP 2. `klue/bert-base` model 및 tokenizer 불러오기

In [7]:
# STEP 2. tokenizer 로드

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)

sample_text = "이 영화는 정말 재미있고 배우들의 연기도 훌륭했습니다."
print(tokenizer.tokenize(sample_text))
print(tokenizer(sample_text))

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Tokenizer: BertTokenizer
Vocab size: 32000
['이', '영화', '##는', '정말', '재미있', '##고', '배우', '##들', '##의', '연기', '##도', '훌륭', '##했', '##습', '##니다', '.']
{'input_ids': [2, 1504, 3771, 2259, 3944, 6001, 2088, 4165, 2031, 2079, 4483, 2119, 5825, 2371, 2219, 3606, 18, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [8]:
# STEP 2-1. token 길이 분석

sample_size = min(5000, len(datasets["train"]))
sample_for_length = datasets["train"].select(range(sample_size))

token_lengths = []
for row in sample_for_length:
    token_lengths.append(len(tokenizer(row[TEXT_COL], truncation=False)["input_ids"]))

length_series = pd.Series(token_lengths)

length_summary = pd.DataFrame({
    "metric": ["mean", "median", "p90", "p95", "p99", "max"],
    "token_length": [
        length_series.mean(),
        length_series.median(),
        length_series.quantile(0.90),
        length_series.quantile(0.95),
        length_series.quantile(0.99),
        length_series.max(),
    ],
})

display(length_summary)

print(f"MAX_LENGTH = {MAX_LENGTH}")
print("128이면 NSMC 대부분 문장을 충분히 담으면서 연산량을 과도하게 키우지 않는 설정입니다.")

,metric,token_length
0,mean,22.4938
1,median,18.0000
2,p90,45.0000
3,p95,61.0000
4,p99,81.0000
5,max,105.0000


MAX_LENGTH = 128
128이면 NSMC 대부분 문장을 충분히 담으면서 연산량을 과도하게 키우지 않는 설정입니다.


# STEP 3. tokenizer 기반 데이터셋 전처리

In [9]:
# STEP 3-1. STEP 4용 고정 padding 전처리
# 모든 문장을 MAX_LENGTH까지 고정 padding합니다.
# 장점: 단순함
# 단점: 짧은 문장도 128 토큰으로 계산되어 padding 낭비가 큼

def tokenize_static_padding(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

tokenized_static = datasets.map(
    tokenize_static_padding,
    batched=True,
    remove_columns=[TEXT_COL],
    desc="Tokenizing with static max_length padding",
)

tokenized_static.set_format("torch")

print(tokenized_static)
print(tokenized_static["train"][0].keys())

Tokenizing with static max_length padding:   0%|          | 0/146182 [00:00<?, ? examples/s]

Tokenizing with static max_length padding:   0%|          | 0/49157 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 146182
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 49157
    })
})
dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask'])


In [10]:
# STEP 3-2. STEP 5용 dynamic padding/bucketing 전처리
# 여기서는 padding을 하지 않고 input_ids 길이만 저장합니다.
# 실제 padding은 batch가 만들어지는 순간 DataCollatorWithPadding이 처리합니다.

def tokenize_dynamic_padding(batch):
    encoded = tokenizer(
        batch[TEXT_COL],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH,
    )
    encoded["length"] = [len(ids) for ids in encoded["input_ids"]]
    return encoded

tokenized_dynamic = datasets.map(
    tokenize_dynamic_padding,
    batched=True,
    remove_columns=[TEXT_COL],
    desc="Tokenizing without padding for dynamic padding/bucketing",
)

tokenized_dynamic.set_format("torch")

data_collator_dynamic = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None,
)

print(tokenized_dynamic)
print(tokenized_dynamic["train"][0].keys())

Tokenizing without padding for dynamic padding/bucketing:   0%|          | 0/146182 [00:00<?, ? examples/s]

Tokenizing without padding for dynamic padding/bucketing:   0%|          | 0/49157 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask', 'length'],
        num_rows: 146182
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask', 'length'],
        num_rows: 49157
    })
})
dict_keys(['labels', 'input_ids', 'token_type_ids', 'attention_mask', 'length'])


# 공통 학습 함수 정의

아래 함수는 STEP 4와 STEP 5를 동일 조건에서 비교하기 위해 사용합니다.

In [11]:
# 공통 metric 함수

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="binary")

    return {
        "accuracy": acc,
        "f1": f1,
    }


def build_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )
    return model


def make_training_args(output_dir: str, group_by_length: bool = False):
    # transformers 버전에 따라 evaluation_strategy/eval_strategy 이름이 다를 수 있어 자동 대응합니다.
    signature = inspect.signature(TrainingArguments.__init__).parameters

    args_dict = {
        "output_dir": output_dir,
        "overwrite_output_dir": True,
        "num_train_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "logging_strategy": "steps",
        "logging_steps": 500,
        "save_strategy": "epoch",
        "save_total_limit": 2,
        "load_best_model_at_end": True,
        "metric_for_best_model": "accuracy",
        "greater_is_better": True,
        "report_to": "none",
        "seed": SEED,
        "data_seed": SEED,
        "fp16": torch.cuda.is_available(),
        "group_by_length": group_by_length,
    }

    if group_by_length:
        args_dict["length_column_name"] = "length"

    if "eval_strategy" in signature:
        args_dict["eval_strategy"] = "epoch"
    else:
        args_dict["evaluation_strategy"] = "epoch"

    # 현재 설치된 transformers가 지원하는 인자만 넘깁니다.
    args_dict = {k: v for k, v in args_dict.items() if k in signature}

    return TrainingArguments(**args_dict)


def build_trainer(
    model,
    args,
    train_dataset,
    eval_dataset,
    data_collator=None,
):
    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": train_dataset,
        "eval_dataset": eval_dataset,
        "data_collator": data_collator,
        "compute_metrics": compute_metrics,
        "callbacks": [
            EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)
        ],
    }

    # 최신 transformers는 tokenizer 대신 processing_class를 권장합니다.
    trainer_signature = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in trainer_signature:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    return Trainer(**trainer_kwargs)


def train_and_evaluate(
    run_name: str,
    tokenized_dataset,
    output_dir: str,
    group_by_length: bool = False,
    data_collator=None,
):
    print("=" * 80)
    print(f"RUN: {run_name}")
    print("=" * 80)

    set_seed_all(SEED)

    model = build_model()
    args = make_training_args(
        output_dir=output_dir,
        group_by_length=group_by_length,
    )

    trainer = build_trainer(
        model=model,
        args=args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        data_collator=data_collator,
    )

    start_time = time.perf_counter()
    train_result = trainer.train()
    train_time_sec = time.perf_counter() - start_time

    eval_metrics = trainer.evaluate()

    result = {
        "run_name": run_name,
        "train_time_sec": train_time_sec,
        "train_time_min": train_time_sec / 60,
        "eval_accuracy": eval_metrics.get("eval_accuracy"),
        "eval_f1": eval_metrics.get("eval_f1"),
        "eval_loss": eval_metrics.get("eval_loss"),
        "train_loss": train_result.training_loss,
        "global_step": train_result.global_step,
        "epoch": eval_metrics.get("epoch"),
        "group_by_length": group_by_length,
        "max_length": MAX_LENGTH,
        "epochs_setting": NUM_EPOCHS,
        "lr": LEARNING_RATE,
        "train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    }

    print(json.dumps(result, indent=2, ensure_ascii=False))

    return trainer, result

# STEP 4. Fine-tuning: 고정 padding 방식

이 단계는 `padding="max_length"`를 사용합니다.

In [12]:
# STEP 4. Fine-tuning 실행: static padding

trainer_static, result_static = train_and_evaluate(
    run_name="STEP4_static_max_length_padding",
    tokenized_dataset=tokenized_static,
    output_dir=STATIC_OUTPUT_DIR,
    group_by_length=False,
    data_collator=None,
)

# 메모리 정리
static_eval_accuracy = result_static["eval_accuracy"]
static_eval_f1 = result_static["eval_f1"]

del trainer_static
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

RUN: STEP4_static_max_length_padding


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_rat

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.259692,0.257682,0.896414,0.893419
2,0.187518,0.243600,0.904998,0.906916
3,0.130062,0.301881,0.906158,0.907200
4,0.077124,0.381758,0.904571,0.905437


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.077124,0.301881,4,0.906158,0.907200


{
  "run_name": "STEP4_static_max_length_padding",
  "train_time_sec": 3985.1691614360006,
  "train_time_min": 66.41948602393335,
  "eval_accuracy": 0.9061578208596944,
  "eval_f1": 0.9071999034380092,
  "eval_loss": 0.30188071727752686,
  "train_loss": 0.17762525714599584,
  "global_step": 18276,
  "epoch": null,
  "group_by_length": false,
  "max_length": 128,
  "epochs_setting": 4,
  "lr": 2e-05,
  "train_batch_size": 32,
  "eval_batch_size": 64
}


# STEP 5. Bucketing + Dynamic Padding 적용 학습

이 단계는 아래 두 가지를 적용합니다.

1. `DataCollatorWithPadding`: batch 내부 최장 길이에 맞춰 동적 padding
2. `TrainingArguments(group_by_length=True)`: 비슷한 길이의 샘플끼리 batch를 구성하여 padding 낭비 감소

In [13]:
# STEP 5. Fine-tuning 실행: bucketing + dynamic padding

trainer_bucket, result_bucket = train_and_evaluate(
    run_name="STEP5_dynamic_padding_group_by_length",
    tokenized_dataset=tokenized_dynamic,
    output_dir=BUCKET_OUTPUT_DIR,
    group_by_length=True,
    data_collator=data_collator_dynamic,
)

RUN: STEP5_dynamic_padding_group_by_length


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_rat

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.258467,0.264334,0.892365,0.887624
2,0.189023,0.237937,0.906809,0.907970
3,0.132464,0.295800,0.906321,0.907391
4,0.081811,0.376541,0.904897,0.906205


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.081811,0.237937,4,0.906809,0.907970


{
  "run_name": "STEP5_dynamic_padding_group_by_length",
  "train_time_sec": 2898.0345649560004,
  "train_time_min": 48.30057608260001,
  "eval_accuracy": 0.9068087963057143,
  "eval_f1": 0.9079695441669847,
  "eval_loss": 0.2379370927810669,
  "train_loss": 0.17762102666426133,
  "global_step": 18276,
  "epoch": null,
  "group_by_length": true,
  "max_length": 128,
  "epochs_setting": 4,
  "lr": 2e-05,
  "train_batch_size": 32,
  "eval_batch_size": 64
}


# STEP 5-1. STEP 4와 STEP 5 결과 비교

비교 기준:

- validation accuracy
- validation f1
- validation loss
- train time
- 속도 개선율
- 정확도 차이

In [14]:
# STEP 5-1. 결과 비교표 생성

comparison_df = pd.DataFrame([result_static, result_bucket])

static_time = comparison_df.loc[comparison_df["run_name"] == "STEP4_static_max_length_padding", "train_time_sec"].iloc[0]
bucket_time = comparison_df.loc[comparison_df["run_name"] == "STEP5_dynamic_padding_group_by_length", "train_time_sec"].iloc[0]

static_acc = comparison_df.loc[comparison_df["run_name"] == "STEP4_static_max_length_padding", "eval_accuracy"].iloc[0]
bucket_acc = comparison_df.loc[comparison_df["run_name"] == "STEP5_dynamic_padding_group_by_length", "eval_accuracy"].iloc[0]

comparison_df["speedup_vs_static_pct"] = (static_time - comparison_df["train_time_sec"]) / static_time * 100
comparison_df["accuracy_gap_vs_static"] = comparison_df["eval_accuracy"] - static_acc

display(comparison_df)

best_acc = comparison_df["eval_accuracy"].max()
best_row = comparison_df.loc[comparison_df["eval_accuracy"].idxmax()]

print("=" * 80)
print("최종 요약")
print("=" * 80)
print(f"Static padding accuracy: {static_acc:.4f}")
print(f"Bucketing/dynamic padding accuracy: {bucket_acc:.4f}")
print(f"Best run: {best_row['run_name']}")
print(f"Best validation accuracy: {best_acc:.4f}")

if best_acc >= 0.90:
    print("판정: validation accuracy 90% 이상 조건 충족")
else:
    print("판정: 90% 미만입니다.")
    print("개선 방법: NUM_EPOCHS=5, LEARNING_RATE=1.5e-5 또는 2e-5 재시도, batch size 조정, seed 변경을 검토하세요.")

if bucket_time < static_time:
    print(f"Bucketing/dynamic padding은 static padding 대비 학습 시간이 약 {(static_time - bucket_time) / static_time * 100:.2f}% 감소했습니다.")
else:
    print("이번 실행에서는 bucketing/dynamic padding의 시간 이점이 크지 않았습니다. 하드웨어/배치 구성/문장 길이 분포 영향을 확인하세요.")

print()
print("분석 결론:")
print("- bucketing/dynamic padding은 모델 구조나 라벨 정보를 바꾸지 않습니다.")
print("- 따라서 성능 향상보다는 padding 토큰 계산량 감소로 인한 학습 속도 개선이 핵심 이점입니다.")
print("- accuracy 차이는 보통 작고, seed/mini-batch 순서/학습 변동성 때문에 ±0.1~0.3%p 정도 흔들릴 수 있습니다.")
print("- 만약 속도는 빨라졌지만 accuracy가 거의 같다면, bucketing의 목적은 제대로 달성된 것입니다.")

,run_name,train_time_sec,train_time_min,eval_accuracy,eval_f1,eval_loss,train_loss,global_step,epoch,group_by_length,max_length,epochs_setting,lr,train_batch_size,eval_batch_size,speedup_vs_static_pct,accuracy_gap_vs_static
0,STEP4_static_max_length_padding,3985.169161,66.419486,0.906158,0.90720,0.301881,0.177625,18276,None,False,128,4,0.00002,32,64,0.000000,0.000000
1,STEP5_dynamic_padding_group_by_length,2898.034565,48.300576,0.906809,0.90797,0.237937,0.177621,18276,None,True,128,4,0.00002,32,64,27.279509,0.000651


최종 요약
Static padding accuracy: 0.9062
Bucketing/dynamic padding accuracy: 0.9068
Best run: STEP5_dynamic_padding_group_by_length
Best validation accuracy: 0.9068
판정: validation accuracy 90% 이상 조건 충족
Bucketing/dynamic padding은 static padding 대비 학습 시간이 약 27.28% 감소했습니다.

분석 결론:
- bucketing/dynamic padding은 모델 구조나 라벨 정보를 바꾸지 않습니다.
- 따라서 성능 향상보다는 padding 토큰 계산량 감소로 인한 학습 속도 개선이 핵심 이점입니다.
- accuracy 차이는 보통 작고, seed/mini-batch 순서/학습 변동성 때문에 ±0.1~0.3%p 정도 흔들릴 수 있습니다.
- 만약 속도는 빨라졌지만 accuracy가 거의 같다면, bucketing의 목적은 제대로 달성된 것입니다.


# 모델 정상 작동 확인: 샘플 문장 예측

In [15]:
# 최종 모델 저장

FINAL_MODEL_DIR = "./nsmc-klue-bert-final-bucketing"

trainer_bucket.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print(f"saved to: {FINAL_MODEL_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to: ./nsmc-klue-bert-final-bucketing


In [16]:
# 모델 정상 작동 확인용 추론 함수

def predict_sentiment(texts):
    model = trainer_bucket.model
    model.eval()

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()

    preds = probs.argmax(axis=-1)

    rows = []
    for text, pred, prob in zip(texts, preds, probs):
        rows.append({
            "text": text,
            "pred_label_id": int(pred),
            "pred_label": ID2LABEL[int(pred)],
            "negative_prob": float(prob[0]),
            "positive_prob": float(prob[1]),
        })

    return pd.DataFrame(rows)


test_texts = [
    "이 영화는 정말 재미있고 배우들의 연기도 훌륭했습니다.",
    "시간이 아까울 정도로 지루하고 별로였습니다.",
    "초반은 별로였지만 후반부가 꽤 감동적이었습니다.",
    "스토리도 엉망이고 연출도 너무 산만합니다.",
    "다시 보고 싶을 만큼 만족스러운 영화였습니다.",
]

pred_df = predict_sentiment(test_texts)
display(pred_df)

,text,pred_label_id,pred_label,negative_prob,positive_prob
0,이 영화는 정말 재미있고 배우들의 연기도 훌륭했습니다.,1,POSITIVE,0.016737,0.983263
1,시간이 아까울 정도로 지루하고 별로였습니다.,0,NEGATIVE,0.998475,0.001525
2,초반은 별로였지만 후반부가 꽤 감동적이었습니다.,1,POSITIVE,0.022095,0.977905
3,스토리도 엉망이고 연출도 너무 산만합니다.,0,NEGATIVE,0.998620,0.001380
4,다시 보고 싶을 만큼 만족스러운 영화였습니다.,1,POSITIVE,0.009358,0.990642


# 제출용 분석 문장 자동 생성

In [17]:
# 과제 보고서에 그대로 붙일 수 있는 분석 문장 생성

static_row = comparison_df[comparison_df["run_name"] == "STEP4_static_max_length_padding"].iloc[0]
bucket_row = comparison_df[comparison_df["run_name"] == "STEP5_dynamic_padding_group_by_length"].iloc[0]

accuracy_diff = bucket_row["eval_accuracy"] - static_row["eval_accuracy"]
time_diff_pct = (static_row["train_time_sec"] - bucket_row["train_time_sec"]) / static_row["train_time_sec"] * 100

report_text = f'''
[실험 결과 요약]

본 실험에서는 Hugging Face의 NSMC 데이터셋을 사용하여 klue/bert-base 모델을 이진 감성 분류 모델로 fine-tuning하였다.
NSMC 데이터는 document 컬럼을 입력 문장으로 사용하고, labels 컬럼을 정답 라벨로 사용하였다.
라벨 0은 부정, 라벨 1은 긍정을 의미한다.

STEP 4에서는 모든 입력을 MAX_LENGTH={MAX_LENGTH}로 고정 padding하여 학습하였다.
그 결과 validation accuracy는 {static_row["eval_accuracy"]:.4f}, validation F1은 {static_row["eval_f1"]:.4f}, 학습 시간은 {static_row["train_time_min"]:.2f}분이었다.

STEP 5에서는 DataCollatorWithPadding을 사용해 batch 단위 dynamic padding을 적용하고,
TrainingArguments의 group_by_length=True 옵션을 사용하여 bucketing을 적용하였다.
그 결과 validation accuracy는 {bucket_row["eval_accuracy"]:.4f}, validation F1은 {bucket_row["eval_f1"]:.4f}, 학습 시간은 {bucket_row["train_time_min"]:.2f}분이었다.

두 방식의 accuracy 차이는 {accuracy_diff:+.4f}였고, bucketing/dynamic padding의 학습 시간 변화율은 {time_diff_pct:+.2f}%였다.
bucketing과 dynamic padding은 모델 구조나 데이터 라벨을 바꾸는 기법이 아니라 padding 토큰 계산 낭비를 줄이는 기법이다.
따라서 모델 성능 자체가 크게 상승하기보다는, 유사한 정확도를 유지하면서 학습 시간을 줄이는 것이 핵심 이점이다.

최종적으로 최고 validation accuracy가 {best_acc:.4f}이므로,
{("validation accuracy 90% 이상 조건을 충족하였다." if best_acc >= 0.90 else "validation accuracy 90% 이상 조건은 아직 충족하지 못하였다. 추가 epoch 또는 learning rate 조정이 필요하다.")}
'''

print(report_text)


[실험 결과 요약]

본 실험에서는 Hugging Face의 NSMC 데이터셋을 사용하여 klue/bert-base 모델을 이진 감성 분류 모델로 fine-tuning하였다.
NSMC 데이터는 document 컬럼을 입력 문장으로 사용하고, labels 컬럼을 정답 라벨로 사용하였다.
라벨 0은 부정, 라벨 1은 긍정을 의미한다.

STEP 4에서는 모든 입력을 MAX_LENGTH=128로 고정 padding하여 학습하였다.
그 결과 validation accuracy는 0.9062, validation F1은 0.9072, 학습 시간은 66.42분이었다.

STEP 5에서는 DataCollatorWithPadding을 사용해 batch 단위 dynamic padding을 적용하고,
TrainingArguments의 group_by_length=True 옵션을 사용하여 bucketing을 적용하였다.
그 결과 validation accuracy는 0.9068, validation F1은 0.9080, 학습 시간은 48.30분이었다.

두 방식의 accuracy 차이는 +0.0007였고, bucketing/dynamic padding의 학습 시간 변화율은 +27.28%였다.
bucketing과 dynamic padding은 모델 구조나 데이터 라벨을 바꾸는 기법이 아니라 padding 토큰 계산 낭비를 줄이는 기법이다.
따라서 모델 성능 자체가 크게 상승하기보다는, 유사한 정확도를 유지하면서 학습 시간을 줄이는 것이 핵심 이점이다.

최종적으로 최고 validation accuracy가 0.9068이므로,
validation accuracy 90% 이상 조건을 충족하였다.

